<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: 'Segoe UI', Tahoma, Arial, sans-serif; font-size: 16px;">

# 🎓 دوره جامع مهندسی مالی تصادفی با پایتون (Stochastic Finance)
**بر اساس کتاب Stochastic Finance with Python (Avishek Nag, 2024)**

---
## 🗺️ نقشه راه و فهرست جامع دوره
این دوره شما را از دریافت داده‌های خام بازار تا پیشرفته‌ترین الگوریتم‌های هوش مصنوعی مالی و بهینه‌سازی سبد سهام هدایت می‌کند. با کلیک روی هر لینک می‌توانید به نوت‌بوک مربوط به آن فصل منتقل شوید:

1. **[فصل ۱: معماری نرم‌افزار و زیرساخت داده (شما اینجا هستید)](#)**
   * *مفهوم:* پیاده‌سازی الگوهای شی‌گرا (Adapter) برای دریافت ایزوله و پایدار داده‌های سری زمانی از بازار.
2. **[فصل ۲: مبانی بازارهای مالی و بازده](Fasl_2_Masterclass.ipynb)**
   * *مفهوم:* تبدیل قیمت‌های ناایستا به بازده‌های ایستا (Simple & Log Returns) و بررسی پدیده خوشه‌بندی نوسانات.
3. **[فصل ۳: توزیع‌های احتمال و برآورد پارامتر](Fasl_3_Masterclass.ipynb)**
   * *مفهوم:* توابع مشخصه (CF)، روش حداکثر درستی‌نمایی (MLE) و آپدیت پیوسته باورها با استنتاج بیزی.
4. **[فصل ۴: شبیه‌سازی تصادفی و مونت‌کارلو](Fasl_4_Masterclass.ipynb)**
   * *مفهوم:* روش پذیرش-رد (Acceptance-Rejection) و تکنیک‌های کاهش واریانس در شبیه‌سازی‌های سنگین.
5. **[فصل ۵: فرآیندهای تصادفی](Fasl_5_Masterclass.ipynb)**
   * *مفهوم:* قدم‌زدن تصادفی، همگرایی به حرکت براونی استاندارد ($W_t$) و فرآیند پواسون.
6. **[فصل ۶: مدل‌های دیفیوژن و GBM](Fasl_6_Masterclass.ipynb)**
   * *مفهوم:* حل معادلات دیفرانسیل تصادفی (SDE) با لم ایتو و بک‌تست مدل روی S&P 500.
7. **[فصل ۷: مدل‌های پرشی (Jump Models)](Fasl_7_Masterclass.ipynb)**
   * *مفهوم:* ترکیب دیفیوژن با شوک‌های ناگهانی (مدل مرتون و کو) و بازیابی چگالی با روش کسینوسی (COS).
8. **[فصل ۸: اختیار معامله و بلک-شولز](Fasl_8_Masterclass.ipynb)**
   * *مفهوم:* قیمت‌گذاری آپشن‌ها، ارزش‌گذاری خنثی نسبت به ریسک و مصورسازی سه‌بعدی حساسیت‌ها (یونانی‌ها).
9. **[فصل ۹: معادلات دیفرانسیل جزئی (FDM)](Fasl_9_Masterclass.ipynb)**
   * *مفهوم:* حل مستقیم معادله بلک-شولز با روش ضمنی (Implicit) روی شبکه‌های محاسباتی.
10. **[فصل ۱۰: بهینه‌سازی سبد سرمایه‌گذاری](Fasl_10_Masterclass.ipynb)**
    * *مفهوم:* تئوری مدرن پورتفولیو (مارکویتز)، ماتریس کوواریانس و استخراج مرز کارا (Efficient Frontier).

</div>

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

---
## فصل ۱: ساختار داده‌های مالی و فیلتراسیون اطلاعات (Data Infrastructure)

### 🎯 هدف این نوت‌بوک
در تئوری احتمالات، جریان اطلاعاتی که در طول زمان به بازار می‌رسد را با یک جبر-سیگما به نام **فیلتراسیون (Filtration - $\mathcal{F}_t$)** نشان می‌دهیم. قیمت یک دارایی نظیر $S_t$ باید نسبت به این فیلتراسیون سازگار (Adapted) باشد؛ یعنی قیمت امروز فقط بر اساس اطلاعات تا زمان $t$ شکل گرفته است.

برای پیاده‌سازی این مفهوم در کامپیوتر، ما نیازمند یک دریافت‌کننده (Data Adapter) هستیم. ما از **الگوی طراحی آداپتور (Adapter Pattern)** استفاده می‌کنیم تا معماری سیستم ما مستقل از منبع داده (Yahoo, Bloomberg و غیره) باشد. ما صرفاً یک قرارداد می‌نویسیم که تضمین کند خروجی همیشه یک ماتریس دو بعدی (زمان و قیمت) خواهد بود.

</div>

In [ ]:
# Install core dependencies
!pip install yfinance pandas numpy matplotlib requests

import pandas as pd
import numpy as np
import yfinance as yf
import enum
from abc import ABC, abstractmethod, ABCMeta

print("\n--- Setup Complete! Core engine libraries loaded. ---")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۱: معماری نرم‌افزار (Abstract Base Classes)

با استفاده از ماژول `abc` در پایتون، ما یک «رابط (Interface)» طراحی می‌کنیم. این رابط بیانگر ویژگی‌های قطعی سیستم ماست. فرقی نمی‌کند در آینده داده‌ها را از چه درگاهی دریافت کنیم، تابع `training_set` و `validation_set` حتماً باید وجود داشته باشند.

</div>

In [ ]:
# --- 1. Abstract Interfaces for Market Filtration ---

class Frequency(enum.Enum):
    """Standard time-steps for stochastic modeling"""
    DAILY = "1d"
    WEEKLY = "1wk"
    MONTHLY = "1mo"

class StockPriceDatasetAdapter(metaclass=ABCMeta):
    """
    Interface to access any data source of stock price quotes.
    Multiple implementations can be made to support different data sources.
    """
    DEFAULT_TICKER = "AAPL"

    @property
    @abstractmethod
    def training_set(self) -> pd.DataFrame:
        """
        Function to get training dataset for a given stock symbol.
        Returns a dataframe with exactly two columns: 'time' and 'stock price'
        """
        pass

    @property
    @abstractmethod
    def validation_set(self) -> pd.DataFrame:
        """
        Function to get validation dataset (used for backtesting later).
        """
        pass

class BaseStockPriceDatasetAdapter(StockPriceDatasetAdapter, ABC):
    """Base adapter providing caching for datasets."""
    def __init__(self, ticker: str = None):
        self._ticker = ticker if ticker else self.DEFAULT_TICKER
        self._training_set = None
        self._validation_set = None

    @abstractmethod
    def _connect_and_prepare(self, date_range: tuple) -> pd.DataFrame:
        pass

    @property
    def training_set(self):
        return self._training_set.copy()

    @property
    def validation_set(self):
        return self._validation_set.copy()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۲: پیاده‌سازی دریافت‌کننده (Concrete Adapter)

اکنون رابط `YahooFinancialsAdapter` را طراحی می‌کنیم. در ریاضیات مالی، اگر شرکتی سود نقدی (Dividend) پرداخت کند یا تجزیه سهام (Stock Split) انجام دهد، قیمت ظاهری روی نمودار افت می‌کند که یک افت ساختگی است و ناشی از مکانیزم بازار نیست.
به همین دلیل، ما همیشه از قیمت بسته شدن تعدیل شده (`Adj Close`) استفاده می‌کنیم تا محاسبات ریاضی ما دچار بایاس (Bias) نشود.

</div>

In [ ]:
# --- 2. Concrete Implementation for Yahoo Finance ---

class YahooFinancialsAdapter(BaseStockPriceDatasetAdapter):
    """
    Concrete implementation of the Adapter pattern bridging our internal
    mathematical requirements with Yahoo Finance's API.
    """
    def __init__(
        self,
        ticker: str = None,
        frequency: Frequency = Frequency.DAILY,
        training_set_date_range: tuple = ("2020-01-01", "2022-01-01"),
        validation_set_date_range: tuple = ("2022-01-01", "2023-12-31")
    ):
        super().__init__(ticker=ticker)
        self._frequency = frequency.value
        # Pre-fetch data into cache
        self._training_set = self._connect_and_prepare(training_set_date_range)
        self._validation_set = self._connect_and_prepare(validation_set_date_range)

    def _connect_and_prepare(self, date_range: tuple) -> pd.DataFrame:
        start_date, end_date = date_range
        # Downloading raw data
        data = yf.download(self._ticker, start=start_date, end=end_date,
                           interval=self._frequency, progress=False)

        # Projecting data onto our strict mathematical space [time, stock price]
        df = pd.DataFrame({
            "time": data.index,
            "stock price": data['Adj Close'].values.flatten()
        })
        return df.dropna()

print("Yahoo Adapter Engine is ready.")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۳: آزمایش و فراخوانی فیلتراسیون $\mathcal{F}_t$

در نهایت، الگوی خود را آزمایش می‌کنیم. ما داده‌های شرکت مایکروسافت (`MSFT`) را استخراج کرده و اطمینان حاصل می‌کنیم که خروجی، آماده تزریق به موتور محاسباتی (در فصل ۲) است.

</div>

In [ ]:
# --- 3. Executing the Data Pipeline ---
def test_market_filtration():
    print("Initializing data adapter for Microsoft (MSFT)...")
    msft_adapter = YahooFinancialsAdapter(
        ticker="MSFT",
        frequency=Frequency.DAILY,
        training_set_date_range=("2022-01-01", "2023-01-01")
    )

    data_feed = msft_adapter.training_set

    print("\n[+] Successful Extraction. Displaying the first 5 observations (S_t):\n")
    display(data_feed.head())

    print(f"\n[i] Total observations adapted: {len(data_feed)}")

test_market_filtration()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

---
### 🏁 پایان فصل ۱
معماری نرم‌افزار ما اکنون قادر است بدون هیچ مشکلی جریان اطلاعات (Filtration) را از بازار به محیط پردازشی پایتون منتقل کند. این پایگاه داده در فصل بعدی (فصل ۲) برای اثبات تئوری **ناایستایی (Non-stationarity)** قیمت‌ها و محاسبه **بازده‌های مالی (Financial Returns)** مورد استفاده قرار می‌گیرد.

**[رفتن به فصل ۲ ➔](Fasl_2_Masterclass.ipynb)**

</div>